# Eksperimen Artikel JTIE
**Automatic Product Image Classification for MSME E-Commerce Catalogs Using Lightweight CNN Architectures**

Penulis: Zulkifli (ORCID 0009-0009-3137-3719)

Cara pakai:
1. Menu **Runtime → Change runtime type → T4 GPU**.
2. Jalankan sel satu per satu dari atas (Shift + Enter), atau **Runtime → Run all**.
3. Semua hasil (tabel, grafik, confusion matrix) tersimpan di folder `hasil/` dan bisa diunduh di sel terakhir.

Perkiraan waktu: 40–70 menit di GPU T4.

## 1. Pengaturan

In [ ]:
# ====== PENGATURAN (boleh diubah) ======
JUMLAH_KELAS      = 10     # jumlah kategori produk
MAKS_CITRA_KELAS  = 400    # citra per kategori
EPOCH             = 15
BATCH_SIZE        = 32
LEARNING_RATE     = 1e-3
SEED              = 42
UKURAN_CITRA      = 224
UJI_AUGMENTASI    = True   # True = juga latih model terbaik tanpa augmentasi (untuk bab pembahasan)
# =======================================

## 2. Instal & impor pustaka

In [ ]:
!pip -q install kagglehub
import os, time, random, shutil, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import kagglehub

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Perangkat:', DEVICE)
os.makedirs('hasil', exist_ok=True)

## 3. Unduh dataset (Fashion Product Images – Small, Kaggle)

In [ ]:
path = kagglehub.dataset_download('paramaggarwal/fashion-product-images-small')
print('Lokasi dataset:', path)
# cari styles.csv dan folder images
csv_path = img_dir = None
for root, dirs, files in os.walk(path):
    if 'styles.csv' in files and csv_path is None: csv_path = os.path.join(root, 'styles.csv')
    if os.path.basename(root) == 'images' and img_dir is None: img_dir = root
print(csv_path, img_dir)

## 4. Siapkan data: pilih kategori, bagi train/val/test 70/15/15

In [ ]:
df = pd.read_csv(csv_path, on_bad_lines='skip')
df['path'] = df['id'].astype(str).apply(lambda i: os.path.join(img_dir, i + '.jpg'))
df = df[df['path'].apply(os.path.exists)]

top = df['articleType'].value_counts().head(JUMLAH_KELAS).index.tolist()
df = df[df['articleType'].isin(top)]
df = df.groupby('articleType', group_keys=False).apply(lambda g: g.sample(min(len(g), MAKS_CITRA_KELAS), random_state=SEED))
KELAS = sorted(top); idx = {k:i for i,k in enumerate(KELAS)}
df['label'] = df['articleType'].map(idx)

train_df, tmp = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=SEED)
val_df, test_df = train_test_split(tmp, test_size=0.50, stratify=tmp['label'], random_state=SEED)

ringkas = pd.DataFrame({'Kategori': KELAS,
    'Train': [ (train_df.label==i).sum() for i in range(len(KELAS))],
    'Val':   [ (val_df.label==i).sum() for i in range(len(KELAS))],
    'Test':  [ (test_df.label==i).sum() for i in range(len(KELAS))]})
ringkas.to_csv('hasil/tabel_dataset.csv', index=False)
print('Total citra:', len(df)); ringkas

In [ ]:
# Contoh citra tiap kategori (untuk Gambar di bab Metode)
fig, ax = plt.subplots(2, 5, figsize=(14, 6))
for a, k in zip(ax.flat, KELAS):
    p = df[df.articleType==k].iloc[0]['path']
    a.imshow(Image.open(p).convert('RGB')); a.set_title(k); a.axis('off')
plt.tight_layout(); plt.savefig('hasil/gambar_contoh_dataset.png', dpi=300); plt.show()

## 5. Preprocessing & augmentasi

In [ ]:
MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]
tf_aug = transforms.Compose([
    transforms.Resize((UKURAN_CITRA, UKURAN_CITRA)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(UKURAN_CITRA, scale=(0.85, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
tf_plain = transforms.Compose([
    transforms.Resize((UKURAN_CITRA, UKURAN_CITRA)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class DS(Dataset):
    def __init__(s, d, tf): s.p, s.y, s.tf = d['path'].tolist(), d['label'].tolist(), tf
    def __len__(s): return len(s.p)
    def __getitem__(s, i): return s.tf(Image.open(s.p[i]).convert('RGB')), s.y[i]

def loaders(augment=True):
    tr = DataLoader(DS(train_df, tf_aug if augment else tf_plain), BATCH_SIZE, shuffle=True, num_workers=2)
    va = DataLoader(DS(val_df, tf_plain), BATCH_SIZE, num_workers=2)
    te = DataLoader(DS(test_df, tf_plain), BATCH_SIZE, num_workers=2)
    return tr, va, te

## 6. Model: MobileNetV3-Small, EfficientNet-B0, ShuffleNetV2 (pretrained ImageNet)

In [ ]:
def buat_model(nama, n):
    if nama == 'MobileNetV3-Small':
        m = models.mobilenet_v3_small(weights='IMAGENET1K_V1'); m.classifier[3] = nn.Linear(m.classifier[3].in_features, n)
    elif nama == 'EfficientNet-B0':
        m = models.efficientnet_b0(weights='IMAGENET1K_V1'); m.classifier[1] = nn.Linear(m.classifier[1].in_features, n)
    elif nama == 'ShuffleNetV2':
        m = models.shufflenet_v2_x1_0(weights='IMAGENET1K_V1'); m.fc = nn.Linear(m.fc.in_features, n)
    return m.to(DEVICE)

def ukuran_mb(m):
    torch.save(m.state_dict(), '_tmp.pt'); s = os.path.getsize('_tmp.pt')/1e6; os.remove('_tmp.pt'); return s

## 7. Fungsi latih & evaluasi (dengan early stopping)

In [ ]:
def latih(nama, augment=True, sabar=4):
    tr, va, te = loaders(augment)
    m = buat_model(nama, len(KELAS))
    opt = torch.optim.Adam(m.parameters(), lr=LEARNING_RATE)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=2)
    lossf = nn.CrossEntropyLoss()
    riw = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
    terbaik, tunggu, bobot = 1e9, 0, None
    for ep in range(EPOCH):
        m.train(); tl, tc, tn = 0, 0, 0
        for x, y in tr:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(); o = m(x); l = lossf(o, y); l.backward(); opt.step()
            tl += l.item()*len(y); tc += (o.argmax(1)==y).sum().item(); tn += len(y)
        m.eval(); vl, vc, vn = 0, 0, 0
        with torch.no_grad():
            for x, y in va:
                x, y = x.to(DEVICE), y.to(DEVICE); o = m(x)
                vl += lossf(o, y).item()*len(y); vc += (o.argmax(1)==y).sum().item(); vn += len(y)
        riw['train_loss'].append(tl/tn); riw['val_loss'].append(vl/vn)
        riw['train_acc'].append(tc/tn);  riw['val_acc'].append(vc/vn)
        sch.step(vl/vn)
        print(f'{nama} | epoch {ep+1:02d} | train acc {tc/tn:.4f} | val acc {vc/vn:.4f} | val loss {vl/vn:.4f}')
        if vl/vn < terbaik: terbaik, tunggu, bobot = vl/vn, 0, {k:v.clone() for k,v in m.state_dict().items()}
        else:
            tunggu += 1
            if tunggu >= sabar: print('Early stopping'); break
    m.load_state_dict(bobot)
    return m, riw, te

def evaluasi(m, te):
    m.eval(); yt, yp = [], []
    with torch.no_grad():
        for x, y in te:
            yp += m(x.to(DEVICE)).argmax(1).cpu().tolist(); yt += y.tolist()
    # waktu inferensi per citra (batch 1, rata-rata 100 kali)
    x1 = torch.randn(1, 3, UKURAN_CITRA, UKURAN_CITRA).to(DEVICE)
    with torch.no_grad():
        for _ in range(10): m(x1)
        if DEVICE=='cuda': torch.cuda.synchronize()
        t = time.time()
        for _ in range(100): m(x1)
        if DEVICE=='cuda': torch.cuda.synchronize()
    ms = (time.time()-t)/100*1000
    p, r, f, _ = precision_recall_fscore_support(yt, yp, average='macro', zero_division=0)
    return {'Accuracy (%)': accuracy_score(yt, yp)*100, 'Precision (%)': p*100, 'Recall (%)': r*100,
            'F1-score (%)': f*100, 'Model size (MB)': ukuran_mb(m),
            'Parameters (M)': sum(q.numel() for q in m.parameters())/1e6, 'Inference (ms/image)': ms}, yt, yp

## 8. Jalankan eksperimen utama (3 model)

In [ ]:
MODEL = ['MobileNetV3-Small', 'EfficientNet-B0', 'ShuffleNetV2']
hasil, riwayat, prediksi = [], {}, {}
for nama in MODEL:
    m, riw, te = latih(nama, augment=True)
    met, yt, yp = evaluasi(m, te)
    met = {'Model': nama, **met}; hasil.append(met)
    riwayat[nama], prediksi[nama] = riw, (yt, yp)
    torch.save(m.state_dict(), f'hasil/model_{nama}.pt')
    print(met)

tabel = pd.DataFrame(hasil).round(2)
tabel.to_csv('hasil/tabel_hasil_utama.csv', index=False)
tabel

## 9. Grafik akurasi & loss (Gambar untuk bab Hasil)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for nama in MODEL:
    r = riwayat[nama]; e = range(1, len(r['val_acc'])+1)
    ax[0].plot(e, r['train_acc'], '--', label=f'{nama} train'); ax[0].plot(e, r['val_acc'], label=f'{nama} val')
    ax[1].plot(e, r['train_loss'], '--', label=f'{nama} train'); ax[1].plot(e, r['val_loss'], label=f'{nama} val')
ax[0].set_title('Accuracy'); ax[0].set_xlabel('Epoch'); ax[1].set_title('Loss'); ax[1].set_xlabel('Epoch')
ax[0].legend(fontsize=7); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig('hasil/gambar_akurasi_loss.png', dpi=300); plt.show()

## 10. Confusion matrix & laporan per kelas model terbaik

In [ ]:
terbaik = tabel.sort_values('F1-score (%)', ascending=False).iloc[0]['Model']
yt, yp = prediksi[terbaik]
cm = confusion_matrix(yt, yp)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(KELAS))); ax.set_xticklabels(KELAS, rotation=45, ha='right')
ax.set_yticks(range(len(KELAS))); ax.set_yticklabels(KELAS)
for i in range(len(KELAS)):
    for j in range(len(KELAS)):
        ax.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=8)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title(f'Confusion Matrix – {terbaik}')
plt.tight_layout(); plt.savefig('hasil/gambar_confusion_matrix.png', dpi=300); plt.show()
lap = classification_report(yt, yp, target_names=KELAS, digits=4)
open('hasil/laporan_per_kelas.txt', 'w').write(lap); print('Model terbaik:', terbaik); print(lap)

## 11. Uji pengaruh augmentasi (opsional, untuk bab Pembahasan)

In [ ]:
if UJI_AUGMENTASI:
    m, riw, te = latih(terbaik, augment=False)
    met, _, _ = evaluasi(m, te)
    aug = tabel[tabel.Model==terbaik].iloc[0]
    banding = pd.DataFrame([
        {'Skenario': 'Dengan augmentasi', 'Accuracy (%)': aug['Accuracy (%)'], 'F1-score (%)': aug['F1-score (%)']},
        {'Skenario': 'Tanpa augmentasi',  'Accuracy (%)': met['Accuracy (%)'], 'F1-score (%)': met['F1-score (%)']}]).round(2)
    banding.to_csv('hasil/tabel_augmentasi.csv', index=False); display(banding)

## 12. Unduh semua hasil

In [ ]:
info = {'kelas': KELAS, 'jumlah_citra': len(df), 'epoch_maks': EPOCH, 'batch': BATCH_SIZE,
        'learning_rate': LEARNING_RATE, 'optimizer': 'Adam', 'seed': SEED, 'model_terbaik': terbaik,
        'gpu': torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'CPU'}
json.dump(info, open('hasil/pengaturan_eksperimen.json', 'w'), indent=2)
for f in os.listdir('hasil'):
    if f.endswith('.pt'): os.remove(os.path.join('hasil', f))   # buang bobot model agar zip kecil
shutil.make_archive('hasil_eksperimen_jtie', 'zip', 'hasil')
from google.colab import files
files.download('hasil_eksperimen_jtie.zip')